Versuch die rthema gruppierten Cluster nochmal jristisch validieren und zusammenstellen zu lassen

In [50]:
import os
import json
import random
from pathlib import Path
from mistralai import Mistral
from time import sleep

In [51]:
#Mistralai Setup
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
model = "open-mistral-nemo"
client = Mistral(api_key=api_key)

#Directory Setup
input_dir = Path("cluster_output_by_rthema")
output_dir = Path("cluster_output_by_rthema/final_cluster_suggestions")
output_dir.mkdir(exist_ok=True)

In [52]:
#New Prompt Function for Mistralai to evaluate and optimize our clusters 
def build_prompt(cluster_dict, rthema):
    prompt = f"""
Du bist ein spezialisiertes Sprachmodell für juristische Wissensmodellierung und Clusterevaluation.

Deine Aufgabe ist es, vorliegende Cluster juristischer Competency Questions (CQs) zu analysieren, zu validieren und im Hinblick auf juristische Kohärenz zu optimieren. Die optimierten Cluster sollen in einem späteren Schritt für die Generierung von Sub-Ontologien genutzt werden, die über ein juristisches Dashboard verfügbar gemacht werden.

Bitte bewerte die Struktur der folgenden Cluster zum rthema: {rthema}.

---

## Ziel:
Identifiziere juristisch konsistente Gruppen von Fragen, schlage die Zusammenlegung redundanter Cluster vor und entferne irrelevante oder unklare Cluster. Das Ergebnis dient der Erstellung semantisch robuster, juristisch sinnvoller Sub-Ontologien.

---

## Deine Aufgaben:

1. Kohärenz prüfen: Enthaltene Fragen sollen semantisch und juristisch zusammenpassen.
2. Zusammenlegung vorschlagen: Wenn zwei Cluster inhaltlich stark überschneiden, führe sie zu einem neuen Cluster zusammen.
3. Eliminierung vorschlagen: Cluster mit unzusammenhängenden, vagen oder redundanten Fragen sollen entfernt werden.
4. Finalstruktur bereitstellen: Gib eine überarbeitete, optimierte Clusterstruktur zurück, die als Grundlage für weitere juristische Verarbeitung geeignet ist.

---

## Input-Cluster aus dem rthema: {rthema}
"""
    for cluster_id, data in cluster_dict.items():
        prompt += f"\n### {cluster_id}\nTop Keywords: {', '.join(data['top_keywords'])}\nBeispielfragen:\n"
        questions = data["questions"]
        if len(questions) <= 8:
            selected_questions = questions
        else:
            selected_questions = random.sample(questions, min(7, len(questions)))

        for q in selected_questions:
            prompt += f"- {q}\n"

    prompt += """
---

## Format der Ausgabe (bitte genau einhalten):

{
  "merge": { "Cluster 3_4": ["Cluster 3", "Cluster 4"] },
  "drop": ["Cluster 9"],
  "final": {
    "Cluster 1": [...],
    "Cluster 2": [...],
    "Cluster 3_4": [...],
    ...
  }
}

- "merge": Zusammengeführte Cluster mit neuem Namen.
- "drop": Cluster, die entfernt werden sollen.
- "final": Finale Liste aller optimierten Cluster.

Wichtig: Liefere ausschließlich die JSON-Antwortstruktur – kein Fließtext, keine Kommentare.
Wichtig: Benutze ausschließlich Cluster-IDs (z.B. "Cluster 3"), keine neuen Bezeichnungen oder Interpretationen.
"""
    return prompt


In [53]:
#Loop over all Cluster Files
for file in input_dir.glob("clusters_*.json"):
    with open(file, "r", encoding="utf-8") as f:
        cluster_data = json.load(f)

    rthema = file.stem.replace("clusters_", "")
    print(f"\n🔍 Sende {rthema} an Mistral...")
    prompt = build_prompt(cluster_data, rthema)

    try:
        response = client.chat.complete(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
        content = response.choices[0].message.content
        output_file = output_dir / f"cluster_feedback_{rthema}.json"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"✅ Gespeichert: {output_file.name}")
    except Exception as e:
        print(f"❌ Fehler bei {rthema}: {e}")

    sleep(5)

print("\n🏁 Fertig – Empfehlungen pro Cluster gespeichert.")


🔍 Sende Wohnungseigentum an Mistral...
✅ Gespeichert: cluster_feedback_Wohnungseigentum.json

🔍 Sende Miete Pacht an Mistral...
✅ Gespeichert: cluster_feedback_Miete Pacht.json

🔍 Sende Erbschaft Schenkung an Mistral...
✅ Gespeichert: cluster_feedback_Erbschaft Schenkung.json

🔍 Sende Kauf Tausch Leasing an Mistral...
✅ Gespeichert: cluster_feedback_Kauf Tausch Leasing.json

🔍 Sende BGBAT an Mistral...
✅ Gespeichert: cluster_feedback_BGBAT.json

🔍 Sende Reisevertrag an Mistral...
✅ Gespeichert: cluster_feedback_Reisevertrag.json

🔍 Sende Werkvertrag an Mistral...
✅ Gespeichert: cluster_feedback_Werkvertrag.json

🔍 Sende Sonstiges Recht an Mistral...
✅ Gespeichert: cluster_feedback_Sonstiges Recht.json

🔍 Sende Sachenrecht an Mistral...
✅ Gespeichert: cluster_feedback_Sachenrecht.json

🔍 Sende Versicherungsrecht an Mistral...
✅ Gespeichert: cluster_feedback_Versicherungsrecht.json

🔍 Sende EU-Recht an Mistral...
✅ Gespeichert: cluster_feedback_EU-Recht.json

🔍 Sende SchuldrechtAT an Mi

In [ ]:
#Chunking to make the code work with larger datasets like ours
def chunk_clusters(cluster_dict, chunk_size=10):
    cluster_ids = list(cluster_dict.keys())
    for i in range(0, len(cluster_ids), chunk_size):
        chunk = {cid: cluster_dict[cid] for cid in cluster_ids[i:i+chunk_size]}
        yield i // chunk_size, chunk

#Prompt Function for Mistralai to evaluate and optimize our Clusters
def build_prompt(cluster_dict, rthema):
    prompt = f"""
Du bist ein spezialisiertes Sprachmodell für juristische Wissensmodellierung und Clusterevaluation.

Deine Aufgabe ist es, vorliegende Cluster juristischer Competency Questions (CQs) zu analysieren, zu validieren und im Hinblick auf juristische Kohärenz zu optimieren. Die optimierten Cluster sollen in einem späteren Schritt für die Generierung von Sub-Ontologien genutzt werden, die über ein juristisches Dashboard verfügbar gemacht werden.

Bitte bewerte die Struktur der folgenden Cluster zum rthema: {rthema}.

---

## Ziel:
Identifiziere juristisch konsistente Gruppen von Fragen, schlage die Zusammenlegung redundanter Cluster vor und entferne irrelevante oder unklare Cluster. Das Ergebnis dient der Erstellung semantisch robuster, juristisch sinnvoller Sub-Ontologien.

---

## Deine Aufgaben:

1. Kohärenz prüfen: Enthaltene Fragen sollen semantisch und juristisch zusammenpassen.
2. Zusammenlegung vorschlagen: Wenn zwei Cluster inhaltlich stark überschneiden, führe sie zu einem neuen Cluster zusammen.
3. Eliminierung vorschlagen: Cluster mit unzusammenhängenden, vagen oder redundanten Fragen sollen entfernt werden.
4. Finalstruktur bereitstellen: Gib eine überarbeitete, optimierte Clusterstruktur zurück, die als Grundlage für weitere juristische Verarbeitung geeignet ist.
5. Achte darauf, dass die Cluster-IDs beibehalten werden und keine neuen Bezeichnungen oder Interpretationen verwendet werden.
{{
  "merge": {{ "Cluster 3_4": ["Cluster 3", "Cluster 4"] }},
  "drop": ["Cluster 9"],
  "final": {{
    "Cluster 1": [...],
    "Cluster 2": [...],
    "Cluster 3_4": [...],
    ...
  }}
}}
Wichtig: Nur output in validem JSON oder txt Format – keine Erklärungen oder Kommentare.
"""
    for cid, data in cluster_dict.items():
        prompt += f"\n#### {cid}\nKeywords: {', '.join(data['top_keywords'])}\n"
        prompt += "Fragen:\n" + "\n".join(f"- {q}" for q in data["questions"][:8]) + "\n"
    return prompt

#Main Loop for Chunking and Processing over all rthema Cluster files
for file in input_dir.glob("clusters_*.json"):
    rthema = file.stem.replace("clusters_", "")
    with open(file, "r", encoding="utf-8") as f:
        cluster_data = json.load(f)

    thema_dir = output_dir / rthema
    thema_dir.mkdir(exist_ok=True)

    for chunk_index, chunk in chunk_clusters(cluster_data, chunk_size=10):
        output_file = thema_dir / f"feedback_chunk_{chunk_index:02}.json"
        if output_file.exists():
            print(f"⏩ Skip {output_file.name} (already exists)")
            continue

        print(f"🔍 {rthema} – Chunk {chunk_index}...")
        prompt = build_prompt(chunk, rthema)

        try:
            response = client.chat.complete(
                model=model,
                messages=[{"role": "user", "content": prompt}]
            )
            content = response.choices[0].message.content
            try:
                json.loads(content)  # Validate
            except json.JSONDecodeError:
                output_file = output_file.with_suffix(".txt")  # fallback
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(content)
            print(f"✅ Saved: {output_file.name}")
        except Exception as e:
            print(f"❌ Fehler bei Chunk {chunk_index}: {e}")

        sleep(5)

print("\n🏁 Done – All cluster chunks evaluated, submitted and saved.")


🔍 Wohnungseigentum – Chunk 0...
✅ Saved: feedback_chunk_00.txt
🔍 Wohnungseigentum – Chunk 1...
✅ Saved: feedback_chunk_01.txt
🔍 Wohnungseigentum – Chunk 2...
✅ Saved: feedback_chunk_02.txt
🔍 Wohnungseigentum – Chunk 3...
✅ Saved: feedback_chunk_03.txt
🔍 Wohnungseigentum – Chunk 4...
✅ Saved: feedback_chunk_04.txt
🔍 Wohnungseigentum – Chunk 5...
✅ Saved: feedback_chunk_05.txt
🔍 Wohnungseigentum – Chunk 6...
✅ Saved: feedback_chunk_06.txt
🔍 Miete Pacht – Chunk 0...
✅ Saved: feedback_chunk_00.txt
🔍 Miete Pacht – Chunk 1...
✅ Saved: feedback_chunk_01.txt
🔍 Miete Pacht – Chunk 2...
✅ Saved: feedback_chunk_02.txt
🔍 Miete Pacht – Chunk 3...
✅ Saved: feedback_chunk_03.txt
🔍 Miete Pacht – Chunk 4...
✅ Saved: feedback_chunk_04.txt
🔍 Miete Pacht – Chunk 5...
✅ Saved: feedback_chunk_05.txt
🔍 Miete Pacht – Chunk 6...
✅ Saved: feedback_chunk_06.txt
🔍 Miete Pacht – Chunk 7...
✅ Saved: feedback_chunk_07.txt
🔍 Miete Pacht – Chunk 8...
✅ Saved: feedback_chunk_08.txt
🔍 Miete Pacht – Chunk 9...
✅ Saved: f